# Capítulo 1 — Análise Exploratória de Dados

**Livro:** começa na página **22** do PDF.

**Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
rng = np.random.default_rng(42)
print("Pronto. pandas", pd.__version__)

## Estimativas de Localização · PDF p. 31

#### Exercício: Média, mediana, aparada e ponderada

Gere as rendas de 1.000 pessoas com `rng.lognormal(8, 1)`. Calcule a média, a mediana, a média aparada em 10% (`stats.trim_mean`) e a média ponderada usando pesos aleatórios. Por que a média fica tão acima da mediana?

In [ ]:
renda = rng.lognormal(8, 1, 1000)
pesos = rng.uniform(0.5, 2, 1000)
print(f"média {renda.mean():.0f} | mediana {np.median(renda):.0f} | aparada {stats.trim_mean(renda, 0.1):.0f} | ponderada {np.average(renda, weights=pesos):.0f}")
# a cauda direita longa (poucos muito ricos) puxa a média para cima

## Mediana e Estimativas Robustas · PDF p. 34

#### Exercício: Robustez a um outlier

Pegue 20 salários em torno de 5.000, calcule média e mediana, depois troque um deles por 1.000.000. Quanto cada estimativa muda?

In [ ]:
s = rng.normal(5000, 500, 20)
antes = (s.mean(), np.median(s)); s[0] = 1_000_000
print([round(x) for x in antes], [round(x) for x in (s.mean(), np.median(s))])

## Desvio-padrão e Estimativas Relacionadas · PDF p. 39

#### Exercício: Desvio-padrão × MAD

Para as rendas lognormais, compare desvio-padrão, desvio absoluto médio e MAD (desvio absoluto mediano, `stats.median_abs_deviation`). Qual é menos afetado pela cauda?

In [ ]:
renda = rng.lognormal(8, 1, 1000)
print(f"dp {renda.std(ddof=1):.0f} | desvio abs. médio {np.mean(np.abs(renda - renda.mean())):.0f} | MAD {stats.median_abs_deviation(renda):.0f}")

## Estimativas Baseadas em Percentis · PDF p. 41

#### Exercício: Percentis e amplitude interquartil

Para os dados de diabetes do scikit-learn (`load_diabetes(as_frame=True, scaled=False)`), calcule os percentis 5, 25, 50, 75 e 95 da variável `bmi` e a amplitude interquartil (IQR).

In [ ]:
from sklearn.datasets import load_diabetes
df = load_diabetes(as_frame=True, scaled=False).frame
p = np.percentile(df["bmi"], [5, 25, 50, 75, 95]); print(p, "IQR =", p[3] - p[1])

## Percentis e Boxplots · PDF p. 45

#### Exercício: Boxplot por grupo

Faça um boxplot do `target` (progressão da doença) por sexo (`sex`) com pandas (`df.boxplot`).

In [ ]:
from sklearn.datasets import load_diabetes
df = load_diabetes(as_frame=True, scaled=False).frame
df.boxplot(column="target", by="sex"); plt.suptitle(""); plt.show()

## Tabela de Frequências e Histogramas · PDF p. 47

#### Exercício: Tabela de frequências e histograma

Divida `bmi` em 8 faixas de mesmo tamanho com `pd.cut`, mostre a contagem de cada faixa (`value_counts`, ordenada pela faixa) e o histograma correspondente.

In [ ]:
from sklearn.datasets import load_diabetes
df = load_diabetes(as_frame=True, scaled=False).frame
print(pd.cut(df["bmi"], 8).value_counts().sort_index())
df["bmi"].plot.hist(bins=8, edgecolor="black"); plt.show()

## Estimativas de Densidade · PDF p. 49

#### Exercício: Densidade (KDE) sobre o histograma

Desenhe o histograma normalizado (`density=True`) do `target` e, por cima, a densidade estimada com `stats.gaussian_kde`.

In [ ]:
from sklearn.datasets import load_diabetes
y = load_diabetes(as_frame=True, scaled=False).frame["target"]
plt.hist(y, bins=25, density=True, alpha=0.5)
xs = np.linspace(y.min(), y.max(), 200); plt.plot(xs, stats.gaussian_kde(y)(xs), lw=2); plt.show()

## Explorando Dados Binários e Categóricos · PDF p. 51

#### Exercício: Proporções e gráfico de barras

Simule 500 atrasos de voo com causas `["companhia", "clima", "tráfego", "segurança"]` e probabilidades `[0.4, 0.15, 0.4, 0.05]`. Mostre a proporção de cada causa e um gráfico de barras.

In [ ]:
causas = pd.Series(rng.choice(["companhia", "clima", "tráfego", "segurança"], 500, p=[0.4, 0.15, 0.4, 0.05]))
prop = causas.value_counts(normalize=True); print(prop.round(3))
prop.plot.bar(); plt.show()

## Valor Esperado · PDF p. 54

#### Exercício: Valor esperado de uma oferta

Um serviço tem 5% dos clientes no plano de R$ 300/mês, 15% no de R$ 50 e 80% gratuitos. Calcule o valor esperado por cliente e confira com 100.000 clientes simulados.

In [ ]:
valores, probs = np.array([300, 50, 0]), np.array([0.05, 0.15, 0.80])
print((valores * probs).sum(), rng.choice(valores, 100_000, p=probs).mean().round(2))

## Correlação · PDF p. 55

#### Exercício: Matriz de correlação

Para as variáveis `age, bmi, bp, s1, s5, target` dos dados de diabetes, mostre a matriz de correlação de Pearson e a de Spearman e desenhe a de Pearson com `plt.imshow`.

In [ ]:
from sklearn.datasets import load_diabetes
df = load_diabetes(as_frame=True, scaled=False).frame[["age", "bmi", "bp", "s1", "s5", "target"]]
c = df.corr(); print(c.round(2)); print(df.corr(method="spearman").round(2))
plt.imshow(c, cmap="RdBu", vmin=-1, vmax=1); plt.xticks(range(6), c.columns); plt.yticks(range(6), c.columns); plt.colorbar(); plt.show()

## Compartimentação Hexagonal e Contornos (Representando Numéricos versus Dados Numéricos) · PDF p. 61

#### Exercício: Hexbin para muitos pontos

Gere 50.000 pares (x normal; y = 2x + ruído) e compare um `scatter` com um `hexbin` (`gridsize=40`).

In [ ]:
x = rng.normal(size=50_000); y = 2 * x + rng.normal(size=50_000)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(x, y, s=1, alpha=0.2); ax[1].hexbin(x, y, gridsize=40, cmap="Blues"); plt.show()

## Duas Variáveis Categóricas · PDF p. 64

#### Exercício: Tabela de contingência

Simule 1.000 empréstimos com nota (A–D) e situação (pago, atrasado, inadimplente), onde notas piores têm mais inadimplência. Monte a tabela cruzada com proporções por linha (`pd.crosstab(..., normalize="index")`).

In [ ]:
nota = rng.choice(list("ABCD"), 1000)
p_inad = pd.Series(nota).map({"A": 0.03, "B": 0.08, "C": 0.15, "D": 0.3}).to_numpy()
u = rng.random(1000)
sit = np.where(u < p_inad, "inadimplente", np.where(u < p_inad + 0.1, "atrasado", "pago"))
print(pd.crosstab(nota, sit, normalize="index").round(3))

## Dados Categóricos e Numéricos · PDF p. 65

#### Exercício: Violino por categoria

Com os dados de vinhos (`load_wine(as_frame=True)`), desenhe o teor alcoólico (`alcohol`) por classe de vinho com `plt.violinplot`.

In [ ]:
from sklearn.datasets import load_wine
df = load_wine(as_frame=True).frame
grupos = [g["alcohol"].to_numpy() for _, g in df.groupby("target")]
plt.violinplot(grupos, showmedians=True); plt.xticks([1, 2, 3], ["classe 0", "classe 1", "classe 2"]); plt.show()